# Module 8, Recitation 1: From Predictions to Prescriptions

<a href="https://colab.research.google.com/github/yuri-spizhovyi-mit/ET6-ML/blob/main/module_8/mod8_rec1.pynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>`

## Data Overview
Today we will look at data from IBM about sales under two promotion schemes. Data Source: modified from https://www.ibm.com/communities/analytics/watson-analytics-blog/marketing-campaign-eff-usec_-fastf/

## Goal
We'd like to determine the best promotion for each store based on the data. In reality, we only know the Sales from the promotion that was actually given. However for the sake of the assignment, we will pretend that we know the true best promotion.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.cluster import KMeans

# PART 1: Prescriptive Method - Predict then Optimize

!git clone https://github.com/yuri-spizhovyi-mit/ET6-ML.git
%cd ET6-ML/module_2
# Load data
promotion_train = pd.read_csv("promotion_train.csv")
promotion_test = pd.read_csv("promotion_test.csv")

In [ ]:
# Explore the data
print(promotion_train.head())

   Unnamed: 0  MarketID MarketSize  LocationID  AgeOfStore  Promotion  Week  \
0           1         1     Medium           1           4          1     1   
1           2         1     Medium           1           4          1     2   
2           3         1     Medium           1           4          1     3   
3           4         1     Medium           1           4          1     4   
4           5         1     Medium           2           5          2     1   

   SalesInThousands  Sales_Prom1  Sales_Prom2  
0             33.73        33.73        34.08  
1             35.67        35.67        35.48  
2             29.03        29.03        29.19  
3             39.25        39.25        46.78  
4             27.81        24.96        27.81  


In [ ]:
print(promotion_train["Promotion"].value_counts())

Promotion
1    252
2    132
Name: count, dtype: int64


In [ ]:
# Convert 'MarketSize' column to numerical categories
label_encoder = LabelEncoder()
promotion_train["MarketSize"] = label_encoder.fit_transform(
    promotion_train["MarketSize"]
)
promotion_test["MarketSize"] = label_encoder.transform(promotion_test["MarketSize"])

In [ ]:
# Separate data into two sets by promotion type
promotion1 = promotion_train[promotion_train["Promotion"] == 1]
promotion2 = promotion_train[promotion_train["Promotion"] == 2]

In [ ]:
# Fit a model for each promotion type
lm_prom1 = LinearRegression().fit(
    promotion1[["MarketSize", "AgeOfStore", "Week"]], promotion1["SalesInThousands"]
)
lm_prom2 = LinearRegression().fit(
    promotion2[["MarketSize", "AgeOfStore", "Week"]], promotion2["SalesInThousands"]
)

# Print model summaries (coefficients and intercept)
print("Promotion 1 Model Coefficients:", lm_prom1.coef_)
print("Promotion 1 Model Intercept:", lm_prom1.intercept_)
print("Promotion 2 Model Coefficients:", lm_prom2.coef_)
print("Promotion 2 Model Intercept:", lm_prom2.intercept_)

Promotion 1 Model Coefficients: [-14.42165315   0.13230938   0.08399308]
Promotion 1 Model Intercept: 66.49107853149489
Promotion 2 Model Coefficients: [-12.25425506   0.21480439   0.55224644]
Promotion 2 Model Intercept: 53.695958474373306


In [ ]:
# What is the "Oracle" promotion, i.e. what we would prescribe if we knew everything?
promotion_test["oracle_sales"] = promotion_test[["Sales_Prom1", "Sales_Prom2"]].max(
    axis=1
)
promotion_test["oracle_benefit"] = (
    promotion_test["oracle_sales"] - promotion_test["SalesInThousands"]
)

# What is the average benefit of the oracle prescription?
print("Average Oracle Benefit:", promotion_test["oracle_benefit"].mean())

Average Oracle Benefit: 2.4824390243902443


In [ ]:
# Let's predict using our linear regression models!
promotion_test["pred_prom1"] = lm_prom1.predict(
    promotion_test[["MarketSize", "AgeOfStore", "Week"]]
)
promotion_test["pred_prom2"] = lm_prom2.predict(
    promotion_test[["MarketSize", "AgeOfStore", "Week"]]
)

# We choose to prescribe whichever has the highest prediction.
promotion_test["prescribe"] = np.where(
    promotion_test["pred_prom1"] > promotion_test["pred_prom2"], 1, 2
)
promotion_test["benefit"] = np.where(
    promotion_test["prescribe"] == 1,
    promotion_test["Sales_Prom1"] - promotion_test["SalesInThousands"],
    promotion_test["Sales_Prom2"] - promotion_test["SalesInThousands"],
)

# What is the average benefit of our prescription?
print("Average Prescriptive Benefit:", promotion_test["benefit"].mean())

Average Prescriptive Benefit: 0.1734146341463413
